# 📚 賢者とユイの読書倶楽部 - eBook自動生成

**使い方：**
1. 「ランタイム」→「すべてのセルを実行」
2. ステップ1でAPIキーを入力
3. ステップ2で**本のタイトルだけ**を入力
4. 著者・カテゴリ・説明は自動で取得されます
5. 自動的にMarkdown・EPUB・Wordが生成されます

In [ ]:
# ステップ0: ライブラリインストール
!pip install -q requests ebooklib Pillow python-docx

In [ ]:
# ステップ1: APIキーを入力してください
import getpass
GEMINI_API_KEY = getpass.getpass('Gemini APIキーを入力してください: ')
print('✅ APIキーを設定しました')

In [ ]:
# ステップ2: 本のタイトルを入力してください
BOOK_TITLE = input('本のタイトル: ')
print(f'📖 対象: {BOOK_TITLE}')

In [ ]:
# ステップ3: Gemini API 呼び出し関数
import requests
import json
import re
import time

GEMINI_URL = 'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-001:generateContent'

SYSTEM_PROMPT = """あなたは「賢者とユイの対話形式」で本の本質を伝える人気ライターです。

## キャラクター設定

**賢者（けんじゃ）**
- 50代の穏やかな老哲学者
- 深い洞察力と広大な知識を持つ
- 難しい概念をシンプルなたとえ話で説明する
- 口癖：「そうじゃな」「面白い視点じゃ」「核心を突いておるな」

**ユイ（ゆい）**
- 20代の好奇心旺盛な読者の代弁者
- 素直な疑問をぶつける
- 読者が「自分も同じこと思ってた！」と共感できる存在
- 口癖：「なるほど！」「えっ、それってどういうことですか？」「つまり〜ってことですね！」

## 執筆ルール
1. 対話形式（賢者とユイの会話）で書く
2. 各章は2000〜3000文字を目安に
3. 具体的な例やたとえ話を豊富に使う
4. 読者が実践できるアクションを含める
5. 著作権に配慮し、「この本が伝えるテーマ・哲学・考え方」を自分の言葉で解説する
6. 直接的な本文引用は避け、エッセンスとインサイトを伝える
"""

_last_call_time = 0.0
CALL_INTERVAL = 5

def call_gemini(prompt, max_tokens=4096):
    global _last_call_time

    elapsed = time.time() - _last_call_time
    if elapsed < CALL_INTERVAL:
        wait = CALL_INTERVAL - elapsed
        print(f'  ⏳ {wait:.0f}秒待機中（レート制限対策）...')
        time.sleep(wait)

    payload = {
        'system_instruction': {'parts': [{'text': SYSTEM_PROMPT}]},
        'contents': [{'role': 'user', 'parts': [{'text': prompt}]}],
        'generationConfig': {'maxOutputTokens': max_tokens, 'temperature': 0.9},
    }

    retry_waits = [60, 120, 180]

    for attempt, wait_on_retry in enumerate([0] + retry_waits):
        if wait_on_retry > 0:
            print(f'  ⚠️  レート制限: {wait_on_retry}秒待機後にリトライ ({attempt}/{len(retry_waits)})...')
            time.sleep(wait_on_retry)

        _last_call_time = time.time()
        r = requests.post(GEMINI_URL, params={'key': GEMINI_API_KEY}, json=payload, timeout=120)

        if r.status_code == 429:
            if attempt < len(retry_waits):
                continue
            r.raise_for_status()

        r.raise_for_status()
        return r.json()['candidates'][0]['content']['parts'][0]['text']

def extract_json(text):
    match = re.search(r'```(?:json)?\s*([\s\S]+?)\s*```', text)
    if match:
        return json.loads(match.group(1))
    match = re.search(r'\{[\s\S]+\}', text)
    if match:
        return json.loads(match.group(0))
    raise ValueError('JSONが見つかりません')

print('✅ 関数を定義しました (モデル: gemini-2.0-flash-001)')

In [ ]:
# ステップ3: タイトルから書籍情報を自動検索
print(f'🔍 書籍情報を検索中: {BOOK_TITLE}...')

lookup_prompt = f"""以下の書籍タイトルについて、実際の書籍情報を調べてJSON形式で回答してください。

タイトル: {BOOK_TITLE}

```json
{{
  "author": "著者名（不明な場合は「不明」）",
  "category": "カテゴリ（ビジネス/自己啓発/投資/心理学/小説/歴史/科学 など）",
  "description": "この本の内容を3〜5文で説明"
}}
```

実在する書籍であれば正確な情報を、不明な場合は推測で構いません。"""

lookup_response = call_gemini(lookup_prompt, max_tokens=512)
book_info = extract_json(lookup_response)

BOOK_AUTHOR = book_info.get('author', '不明')
BOOK_CATEGORY = book_info.get('category', 'ビジネス')
BOOK_DESCRIPTION = book_info.get('description', '')

print(f'✅ 書籍情報を取得しました')
print(f'   著者: {BOOK_AUTHOR}')
print(f'   カテゴリ: {BOOK_CATEGORY}')
print(f'   説明: {BOOK_DESCRIPTION[:80]}...' if len(BOOK_DESCRIPTION) > 80 else f'   説明: {BOOK_DESCRIPTION}')

In [ ]:
# ステップ4: 本の構成を計画
print(f'📚 本の構成を計画中: {BOOK_TITLE}...')

plan_prompt = f"""以下の本について、賢者とユイの対話形式で「要約・解説本」を書きます。

## 対象書籍
- タイトル：{BOOK_TITLE}
- 著者：{BOOK_AUTHOR}
- カテゴリ：{BOOK_CATEGORY}
- 説明：{BOOK_DESCRIPTION or '（説明なし）'}

以下のJSON形式で出力してください：
```json
{{
  "book_title": "【賢者とユイが語る】〇〇の本質",
  "subtitle": "〇〇が教えてくれる人生の知恵",
  "description": "本の説明文（D2D投稿用、300文字程度）",
  "keywords": ["キーワード1", "キーワード2"],
  "chapter_titles": ["第1章タイトル", "第2章タイトル", "第3章タイトル", "第4章タイトル", "第5章タイトル", "第6章タイトル"]
}}
```
章は5〜7章構成にしてください。
"""

plan_response = call_gemini(plan_prompt, max_tokens=2048)
plan = extract_json(plan_response)

print(f'✅ 構成完了！')
print(f'  タイトル: {plan["book_title"]}')
print(f'  サブタイトル: {plan["subtitle"]}')
print(f'  章数: {len(plan["chapter_titles"])} 章')
for i, t in enumerate(plan['chapter_titles'], 1):
    print(f'    第{i}章: {t}')

In [ ]:
# ステップ5: まえがき執筆
print('✍️  まえがきを執筆中...')

foreword_prompt = f"""以下の本の解説書のまえがきを書いてください。

## 対象書籍
- タイトル：{BOOK_TITLE}
- 著者：{BOOK_AUTHOR}

## まえがきの要素
- この本を手に取った読者へのメッセージ
- 賢者とユイというキャラクターの紹介
- この解説本で得られること
- 400〜600文字程度

マークダウン形式で出力してください。
"""

foreword = call_gemini(foreword_prompt)
print('✅ まえがき完了')

In [ ]:
# ステップ6: 各章を執筆
chapters = []
chapter_titles = plan['chapter_titles']
all_chapters_str = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(chapter_titles))

for i, title in enumerate(chapter_titles, 1):
    print(f'✍️  第{i}章「{title}」を執筆中...')
    chapter_prompt = f"""以下の本について、第{i}章を書いてください。

## 対象書籍
- タイトル：{BOOK_TITLE}
- 著者：{BOOK_AUTHOR}

## 章の情報
- 章番号：第{i}章
- 章タイトル：{title}
- 全体の章構成：
{all_chapters_str}

## 注意事項
- 著作権を守るため、本の文章を直接引用しない
- テーマ・哲学・考え方をあなた自身の言葉で解説する
- 賢者とユイの自然な対話で進める
- 2000〜3000文字を目安に

## 出力形式
マークダウン形式で：

**ユイ**：（セリフ）

**賢者**：（セリフ）
"""
    content = call_gemini(chapter_prompt, max_tokens=4096)
    chapters.append({'number': i, 'title': title, 'content': content})
    print(f'  ✅ 第{i}章完了 ({len(content):,}文字)')

print(f'\n✅ 全{len(chapters)}章の執筆完了！')

In [ ]:
# ステップ7: あとがき執筆
print('✍️  あとがきを執筆中...')

afterword_prompt = f"""以下の本の解説書のあとがきを書いてください。

## 対象書籍
- タイトル：{BOOK_TITLE}
- 著者：{BOOK_AUTHOR}
- 全章タイトル：
{all_chapters_str}

## あとがきの要素
- 全章を通じて伝えてきたメッセージの総括
- 読者へのエール・行動喚起
- 400〜600文字程度

マークダウン形式で出力してください。
"""

afterword = call_gemini(afterword_prompt)
print('✅ あとがき完了')

In [ ]:
# ステップ8: Markdownファイルとして保存
import os
from pathlib import Path

safe_title = re.sub(r'[\\/*?:"<>|【】]', '', plan['book_title'])[:50]
output_dir = Path('/content/ebook_output')
output_dir.mkdir(exist_ok=True)

# Markdownテキスト組み立て
lines = []
lines.append(f"# {plan['book_title']}")
lines.append(f"\n**{plan['subtitle']}**")
lines.append(f"\n著者：賢者とユイの読書倶楽部")
lines.append(f"\n---\n")
lines.append(f"## まえがき\n")
lines.append(foreword)
lines.append("\n---\n")
for ch in chapters:
    lines.append(f"## 第{ch['number']}章　{ch['title']}\n")
    lines.append(ch['content'])
    lines.append("\n---\n")
lines.append("## あとがき\n")
lines.append(afterword)

full_text = '\n'.join(lines)
md_path = output_dir / f'{safe_title}.md'
md_path.write_text(full_text, encoding='utf-8')

total_chars = len(full_text)
print(f'✅ Markdown保存: {md_path}')
print(f'   総文字数: {total_chars:,} 文字')

In [ ]:
# ステップ9: EPUBファイル生成
from ebooklib import epub
import html

def md_to_html(text):
    """簡易Markdown→HTML変換"""
    lines = text.split('\n')
    result = []
    for line in lines:
        line = html.escape(line)
        if line.startswith('### '):
            line = f'<h3>{line[4:]}</h3>'
        elif line.startswith('## '):
            line = f'<h2>{line[3:]}</h2>'
        elif line.startswith('# '):
            line = f'<h1>{line[2:]}</h1>'
        elif line.startswith('**') and line.endswith('**'):
            line = f'<p><strong>{line[2:-2]}</strong></p>'
        elif line.strip() == '' or line.strip() == '---':
            line = '<br/>'
        else:
            # **太字** をインライン変換
            line = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', line)
            line = f'<p>{line}</p>'
        result.append(line)
    return '\n'.join(result)

book = epub.EpubBook()
book.set_identifier('id_' + safe_title)
book.set_title(plan['book_title'])
book.set_language('ja')
book.add_author('賢者とユイの読書倶楽部')

# CSSスタイル
style = epub.EpubItem(
    uid='style', file_name='style/main.css', media_type='text/css',
    content=b'body{font-family:serif;line-height:1.8;margin:2em;} h1,h2,h3{margin-top:1.5em;} p{margin:0.5em 0;} strong{font-weight:bold;}'
)
book.add_item(style)

spine = ['nav']
toc = []

# まえがき
fw = epub.EpubHtml(title='まえがき', file_name='foreword.xhtml', lang='ja')
fw.content = f'<html><body><h2>まえがき</h2>{md_to_html(foreword)}</body></html>'
fw.add_item(style)
book.add_item(fw)
spine.append(fw)
toc.append(epub.Link('foreword.xhtml', 'まえがき', 'foreword'))

# 各章
for ch in chapters:
    c = epub.EpubHtml(title=f'第{ch["number"]}章 {ch["title"]}', file_name=f'chapter{ch["number"]}.xhtml', lang='ja')
    c.content = f'<html><body><h2>第{ch["number"]}章　{html.escape(ch["title"])}</h2>{md_to_html(ch["content"])}</body></html>'
    c.add_item(style)
    book.add_item(c)
    spine.append(c)
    toc.append(epub.Link(f'chapter{ch["number"]}.xhtml', f'第{ch["number"]}章 {ch["title"]}', f'ch{ch["number"]}'))

# あとがき
aw = epub.EpubHtml(title='あとがき', file_name='afterword.xhtml', lang='ja')
aw.content = f'<html><body><h2>あとがき</h2>{md_to_html(afterword)}</body></html>'
aw.add_item(style)
book.add_item(aw)
spine.append(aw)
toc.append(epub.Link('afterword.xhtml', 'あとがき', 'afterword'))

book.toc = toc
book.spine = spine
book.add_item(epub.EpubNcx())
book.add_item(epub.EpubNav())

epub_path = output_dir / f'{safe_title}.epub'
epub.write_epub(str(epub_path), book)
print(f'✅ EPUB保存: {epub_path}')
print(f'   サイズ: {epub_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# ステップ10: Word(.docx)ファイル生成
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()

# タイトルページ
title_para = doc.add_heading(plan['book_title'], 0)
title_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
sub_para = doc.add_paragraph(plan['subtitle'])
sub_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_paragraph('著者：賢者とユイの読書倶楽部').alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_page_break()

def add_md_to_doc(doc, text):
    for line in text.split('\n'):
        if line.startswith('## '):
            doc.add_heading(line[3:], level=2)
        elif line.startswith('# '):
            doc.add_heading(line[2:], level=1)
        elif line.strip() == '' or line.strip() == '---':
            doc.add_paragraph('')
        else:
            # **太字** 処理
            p = doc.add_paragraph()
            parts = re.split(r'(\*\*.+?\*\*)', line)
            for part in parts:
                if part.startswith('**') and part.endswith('**'):
                    run = p.add_run(part[2:-2])
                    run.bold = True
                else:
                    p.add_run(part)

# まえがき
doc.add_heading('まえがき', level=1)
add_md_to_doc(doc, foreword)
doc.add_page_break()

# 各章
for ch in chapters:
    doc.add_heading(f'第{ch["number"]}章　{ch["title"]}', level=1)
    add_md_to_doc(doc, ch['content'])
    doc.add_page_break()

# あとがき
doc.add_heading('あとがき', level=1)
add_md_to_doc(doc, afterword)

docx_path = output_dir / f'{safe_title}.docx'
doc.save(str(docx_path))
print(f'✅ Word保存: {docx_path}')
print(f'   サイズ: {docx_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# ステップ11: ファイルをダウンロード
from google.colab import files

print('=' * 50)
print('🎉 成果物が完成しました！')
print('=' * 50)
print(f'📖 タイトル: {plan["book_title"]}')
print(f'📝 総文字数: {total_chars:,} 文字')
print(f'📚 章数: {len(chapters)} 章')
print()
print('ダウンロードを開始します...')

files.download(str(md_path))
files.download(str(epub_path))
files.download(str(docx_path))

print('✅ 3ファイルのダウンロードが完了しました')
print('  - Markdown (.md)')
print('  - EPUB (.epub) ← D2D投稿用')
print('  - Word (.docx) ← Googleドキュメントで編集可')